In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# Experiment Tracking — Ghi lại để tái lập quyết định

Reference thực hành. Dự đoán trước mỗi experiment, rồi ghi Result → Observation → Why.

## Thiết kế run contract

Config, dữ liệu, split và version phải đi cùng score. Lab ghi JSON và lưu pipeline local; chỉ load joblib do chính notebook tạo. Timestamp/latency không dùng làm điều kiện tái lập số.

In [ ]:
from hashlib import sha256
from time import perf_counter
import platform
import sklearn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def content_hash(config):
    """Return SHA256 of canonical JSON; only JSON-serializable values are allowed."""
    encoded = json.dumps(config, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return sha256(encoded).hexdigest()

assert content_hash({'C': 1.0, 'seed': 42}) == content_hash({'seed': 42, 'C': 1.0})
assert content_hash({'C': 1.0}) != content_hash({'C': 0.1})
X = rng.normal(size=(160 if FAST else 400, 4))
y = (X[:, 0] - 0.7 * X[:, 1] + rng.normal(0, 0.3, len(X)) > 0).astype(int)
X_test = rng.normal(size=(30, 4))
test_ids = np.array([f'test_{i}' for i in range(len(X_test))])
tr, va = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)
data_record = {'X': X.tolist(), 'y': y.tolist(), 'X_test': X_test.tolist(), 'test_ids': test_ids.tolist()}
data_hash = content_hash(data_record)
split_record = {'train': tr.tolist(), 'validation': va.tolist()}
split_hash = content_hash(split_record)
assert set(tr).isdisjoint(va)
versions = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'joblib': joblib.__version__}
# This explicit implementation version must change if training logic changes.
code_version = 'module06-tracking-logistic-v1'
np.savez(OUT / 'tracking_data.npz', X=X, y=y, X_test=X_test, ids=test_ids)
(OUT / 'tracking_split.json').write_text(json.dumps(split_record, indent=2), encoding='utf-8')


## Experiment — Một biến mỗi lần

**Hypothesis:** thay C (nghịch đảo regularization strength) có thể đổi F1. Dự đoán trước khi chạy. Giữ split/data/seed cố định. Log schema có status và error để thất bại không biến mất khỏi lịch sử.

In [ ]:
def build_model(config):
    """Build an unfitted pipeline from persisted config."""
    return make_pipeline(StandardScaler(), LogisticRegression(C=config['C'],
        max_iter=config['max_iter'], random_state=config['seed']))

records = []
best_model, best_record = None, None
for C in [0.1, 1.0, 10.0]:
    config = {'C': C, 'seed': 42, 'max_iter': 500, 'fast_mode': FAST}
    identity = {'config': config, 'data_hash': data_hash, 'split_hash': split_hash,
                'code_version': code_version, 'versions': versions}
    record = {'schema_version': 1, 'run_id': content_hash(identity),
              'config_hash': content_hash(config), **identity,
              'metric': 'macro_f1', 'status': 'running', 'score': None}
    start = perf_counter()
    try:
        model = build_model(config).fit(X[tr], y[tr])
        prediction = model.predict(X[va])
        score = f1_score(y[va], prediction, labels=[0, 1], average='macro', zero_division=0)
        record.update(status='passed', score=float(score))
        # WHY: a deterministic tie-break chooses lower C, not noisy measured latency.
        if best_record is None or (-score, C) < (-best_record['score'], best_record['config']['C']):
            best_model, best_record = model, record
    except (ValueError, FloatingPointError) as exc:
        record.update(status='failed', error=f'{type(exc).__name__}: {exc}')
    record['elapsed_seconds'] = perf_counter() - start
    records.append(record)
    (OUT / 'experiments.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
    print('C/score/status:', C, record['score'], record['status'])
    print('runtime seconds:', record['elapsed_seconds'])
assert len(records) == 3 and all(r['status'] == 'passed' for r in records)
assert len({r['run_id'] for r in records}) == 3
assert len({r['split_hash'] for r in records}) == 1
assert best_model is not None
joblib.dump(best_model, OUT / 'tracking_best.joblib')
(OUT / 'tracking_best.json').write_text(json.dumps(best_record, indent=2), encoding='utf-8')
expected = best_model.predict_proba(X[va])
np.save(OUT / 'tracking_expected.npy', expected)
del best_model, model, X, y, X_test, tr, va, expected


## Replay từ artifact

Đọc config, split, data từ disk. So cả loaded model lẫn model refit từ config trong cùng môi trường. Đây là bằng chứng local; chưa chứng minh mọi hardware/phiên bản sẽ cho cùng kết quả.

In [ ]:
record = json.loads((OUT / 'tracking_best.json').read_text(encoding='utf-8'))
splits = json.loads((OUT / 'tracking_split.json').read_text(encoding='utf-8'))
with np.load(OUT / 'tracking_data.npz', allow_pickle=False) as data:
    X, y, X_test, test_ids = data['X'], data['y'], data['X_test'], data['ids']
assert content_hash({'X': X.tolist(), 'y': y.tolist(), 'X_test': X_test.tolist(),
                     'test_ids': test_ids.tolist()}) == record['data_hash']
assert content_hash(splits) == record['split_hash']
assert record['versions'] == versions
assert record['code_version'] == code_version
tr, va = np.array(splits['train']), np.array(splits['validation'])
loaded = joblib.load(OUT / 'tracking_best.joblib')
retrained = build_model(record['config']).fit(X[tr], y[tr])
expected = np.load(OUT / 'tracking_expected.npy', allow_pickle=False)
assert np.allclose(loaded.predict_proba(X[va]), expected, rtol=1e-7, atol=1e-9)
assert np.allclose(retrained.predict_proba(X[va]), expected, rtol=1e-7, atol=1e-9)
test_pred = loaded.predict(X_test)
print('replay passed:', True)
print('selected C:', record['config']['C'])


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_tracking.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_tracking.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Observation → Why

Đổi thứ tự key không đổi hash; đổi C phải đổi config hash dù predictions có thể giống. `code_version` là nhãn thủ công: khi làm dự án thật, lưu Git commit/source hash và mọi thay đổi chưa commit. Latency được ghi để theo dõi, không là tiêu chí equality. Log hiện ghi đè bộ thí nghiệm cố định để Run All tái lập; dùng run directory riêng nếu cần giữ nhiều phiên học.